# DAY 4 - HEALTHCARE PERFORMANCE, TESTING AND RECOVERY

v5 SME-aligned production hardening evidence

In [0]:
from pyspark.sql import functions as F

CATALOG = "capstone_8hospital"
SILVER = f"{CATALOG}.silver_schema"
GOLD = f"{CATALOG}.gold_schema"


### Check required Healthcare tables

In [0]:
required_tables = [
    f"{SILVER}.silver_patient_encounters",
    f"{SILVER}.silver_patient_master",
    f"{SILVER}.silver_department_master",
    f"{SILVER}.silver_bed_master",
    f"{SILVER}.silver_lab_test_events",
    f"{SILVER}.silver_bed_transfer_events",
    f"{GOLD}.fact_patient_encounter",
    f"{GOLD}.fact_lab_test_event",
    f"{GOLD}.fact_bed_transfer_event",
    f"{GOLD}.mart_department_daily",
    f"{GOLD}.mart_bed_utilization"
]

silver_tables = [r.tableName for r in spark.sql(f"SHOW TABLES IN {SILVER}").collect()]
gold_tables = [r.tableName for r in spark.sql(f"SHOW TABLES IN {GOLD}").collect()]
existing = silver_tables + gold_tables

missing = [t.split(".")[-1] for t in required_tables if t.split(".")[-1] not in existing]

display(spark.createDataFrame(
    [("PASS" if not missing else "FAIL",
      len(required_tables) - len(missing),
      len(missing))],
    ["status", "tables_found", "tables_missing"]
))


### Positive regression tests

In [0]:
encounters = spark.table(f"{SILVER}.silver_patient_encounters")
labs = spark.table(f"{SILVER}.silver_lab_test_events")
transfers = spark.table(f"{SILVER}.silver_bed_transfer_events")

positive_tests = [
    ("Patient encounter keys present",
     encounters.filter(F.col("encounter_id").isNotNull()).count() > 0),
    ("Lab event keys present",
     labs.filter(F.col("lab_event_id").isNotNull()).count() > 0),
    ("Transfer keys present",
     transfers.filter(F.col("transfer_id").isNotNull()).count() > 0),
    ("No duplicate encounter keys",
     encounters.groupBy("encounter_id").count().filter("count > 1").count() == 0),
    ("No duplicate lab event keys",
     labs.groupBy("lab_event_id").count().filter("count > 1").count() == 0),
    ("No duplicate transfer keys",
     transfers.groupBy("transfer_id").count().filter("count > 1").count() == 0)
]

display(spark.createDataFrame(
    [(name, "PASS" if ok else "FAIL") for name, ok in positive_tests],
    ["test_name", "status"]
))


### Healthcare domain regression tests

In [0]:
encounters = (
    encounters
    .withColumn("admission_ts", F.to_timestamp("admission_ts"))
    .withColumn("discharge_ts", F.to_timestamp("discharge_ts"))
)

labs = (
    labs
    .withColumn("ordered_ts", F.to_timestamp("ordered_ts"))
    .withColumn("collected_ts", F.to_timestamp("collected_ts"))
    .withColumn("result_ts", F.to_timestamp("result_ts"))
)

transfers = (
    transfers
    .withColumn("transfer_ts", F.to_timestamp("transfer_ts"))
    .withColumn("recorded_ts", F.to_timestamp("recorded_ts"))
)

domain_tests = [
    ("Admission <= discharge",
     encounters.filter(
         F.col("admission_ts").isNotNull() &
         F.col("discharge_ts").isNotNull() &
         (F.col("admission_ts") > F.col("discharge_ts"))
     ).count() == 0),

    ("Lab order <= collection",
     labs.filter(
         F.col("ordered_ts").isNotNull() &
         F.col("collected_ts").isNotNull() &
         (F.col("ordered_ts") > F.col("collected_ts"))
     ).count() == 0),

    ("Lab collection <= result",
     labs.filter(
         F.col("collected_ts").isNotNull() &
         F.col("result_ts").isNotNull() &
         (F.col("collected_ts") > F.col("result_ts"))
     ).count() == 0),

    ("Transfer <= recorded",
     transfers.filter(
         F.col("transfer_ts").isNotNull() &
         F.col("recorded_ts").isNotNull() &
         (F.col("transfer_ts") > F.col("recorded_ts"))
     ).count() == 0),

    ("Actual bed change",
     transfers.filter(
         F.col("from_bed_id").isNotNull() &
         F.col("to_bed_id").isNotNull() &
         (F.col("from_bed_id") == F.col("to_bed_id"))
     ).count() == 0)
]

display(spark.createDataFrame(
    [(name, "PASS" if ok else "FAIL") for name, ok in domain_tests],
    ["test_name", "status"]
))


### Negative tests

In [0]:
negative_tests = [
    ("No duplicate encounter keys",
     encounters.groupBy("encounter_id").count().filter("count > 1").count() == 0),

    ("No negative LOS",
     spark.table(f"{GOLD}.fact_patient_encounter")
     .filter(F.col("length_of_stay_hours") < 0).count() == 0),

    ("No negative lab turnaround",
     spark.table(f"{GOLD}.fact_lab_test_event")
     .filter(F.col("lab_turnaround_minutes") < 0).count() == 0)
]

display(spark.createDataFrame(
    [(name, "PASS" if ok else "FAIL") for name, ok in negative_tests],
    ["negative_test", "status"]
))


### Silver to Gold reconciliation

In [0]:
gold_checks = [
    (
        "Encounter Silver to Gold",
        encounters.select("encounter_id").distinct().count(),
        spark.table(f"{GOLD}.fact_patient_encounter")
        .select("encounter_id").distinct().count()
    ),
    (
        "Lab Silver to Gold",
        labs.select("lab_event_id").distinct().count(),
        spark.table(f"{GOLD}.fact_lab_test_event")
        .select("lab_event_id").distinct().count()
    ),
    (
        "Transfer Silver to Gold",
        transfers.select("transfer_id").distinct().count(),
        spark.table(f"{GOLD}.fact_bed_transfer_event")
        .select("transfer_id").distinct().count()
    )
]

display(spark.createDataFrame(
    [(name, silver_count, gold_count,
      "PASS" if silver_count == gold_count else "FAIL")
     for name, silver_count, gold_count in gold_checks],
    ["check_name", "silver_distinct", "gold_distinct", "status"]
))


### Replay and idempotency check

In [0]:
before = {
    "encounters": spark.table(f"{GOLD}.fact_patient_encounter").count(),
    "labs": spark.table(f"{GOLD}.fact_lab_test_event").count(),
    "transfers": spark.table(f"{GOLD}.fact_bed_transfer_event").count()
}

# Rerun the same pipeline/input, then execute this cell again.
after = {
    "encounters": spark.table(f"{GOLD}.fact_patient_encounter").count(),
    "labs": spark.table(f"{GOLD}.fact_lab_test_event").count(),
    "transfers": spark.table(f"{GOLD}.fact_bed_transfer_event").count()
}

display(spark.createDataFrame(
    [(name, before[name], after[name],
      "PASS" if before[name] == after[name] else "FAIL")
     for name in before],
    ["table_name", "before_replay", "after_replay", "status"]
))


### Checkpoint recovery evidence

In [0]:
checkpoint_locations = [
    "/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/_autoloader_state/bronze/patient_encounters",
    "/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/_autoloader_state/bronze/lab_test_events",
    "/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/_autoloader_state/bronze/bed_transfer_events"
]

checkpoint_status = []

for path in checkpoint_locations:
    try:
        dbutils.fs.ls(path)
        status = "PRESENT"
    except Exception:
        status = "MISSING"
    checkpoint_status.append((path, status))

display(spark.createDataFrame(
    checkpoint_status,
    ["checkpoint_path", "status"]
))


### Unity Catalog grants

In [0]:
display(spark.sql(f"SHOW GRANTS ON SCHEMA {SILVER}"))
display(spark.sql(f"SHOW GRANTS ON SCHEMA {GOLD}"))


### Delta history

In [0]:
display(
    spark.sql(f"DESCRIBE HISTORY {GOLD}.mart_department_daily").limit(10)
)


### OPTIMIZE performance benchmark

In [0]:
benchmark_table = f"{GOLD}.mart_department_daily"

before_count = spark.table(benchmark_table).count()

spark.sql(f"OPTIMIZE {benchmark_table}")

after_count = spark.table(benchmark_table).count()

display(spark.createDataFrame(
    [(
        benchmark_table,
        before_count,
        after_count,
        "PASS" if before_count == after_count else "FAIL"
    )],
    [
        "table_name",
        "rows_before_optimize",
        "rows_after_optimize",
        "row_integrity"
    ]
))


### Monitoring summary

In [0]:
monitoring = [
    ("Gold encounter fact",
     spark.table(f"{GOLD}.fact_patient_encounter").count()),
    ("Gold lab fact",
     spark.table(f"{GOLD}.fact_lab_test_event").count()),
    ("Gold transfer fact",
     spark.table(f"{GOLD}.fact_bed_transfer_event").count()),
    ("Department KPI rows",
     spark.table(f"{GOLD}.mart_department_daily").count())
]

display(spark.createDataFrame(
    monitoring,
    ["metric", "value"]
))


### Incident runbook

1. Check the failed Workflow task.
2. Check task dependencies and retry status.
3. Check the Auto Loader checkpoint for the affected source.
4. Check Delta history for the affected table.
5. Rerun the failed task or replay the source batch.
6. Confirm that no duplicate business keys were introduced.
7. Run regression and reconciliation checks again.
8. Record the failure, recovery action and final status.

### Day 4 evidence status

This notebook provides notebook-level evidence for healthcare regression, reconciliation, replay/idempotency checks, checkpoint inspection, Unity Catalog grants, Delta history, query plan, OPTIMIZE and monitoring.

Workflow DAG configuration, retry/alert configuration and Git PR must be evidenced from the Databricks Workflow/Git UI. Do not mark those items complete from notebook code alone.